In [ ]:
pip install pandas_ta

In [ ]:
import math
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas_ta as ta

In [ ]:
pip install torch_geometric

In [ ]:
# Install and import PyG modules separately; user must have torch-geometric installed.
try:
    from torch_geometric.nn import GCNConv
except Exception as e:
    raise ImportError("Please install torch-geometric following the official instructions. Error: " + str(e))

# Transformers FinBERT
from transformers import pipeline

# yfinance for price data
import yfinance as yf

### Apple

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info.xlsx", sheet_name="Apple")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

### HSBC

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info_part2.xlsx", sheet_name="HSBC")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

### Pepsi

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info_part2.xlsx", sheet_name="Pepsi")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

### Toyota

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info.xlsx", sheet_name="Toyota")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

### Tencent

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info.xlsx", sheet_name="Tencent")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

In [ ]:
import math
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

### TGAT

In [ ]:
from torch_geometric.nn import GATConv

In [ ]:
# -------------------------
# Utilities
# -------------------------
def returns_from_prices(prices: np.ndarray) -> np.ndarray:
    """Compute log returns. prices shape (T, N) -> returns (T-1, N)"""
    return np.log(prices[1:] / prices[:-1] + 1e-12)


def finbert_sentiment_pipeline(model_name: str = "yiyanghkust/finbert-tone", device: int = -1):
    """
    Create a transformers pipeline for FinBERT sentiment analysis.
    device=-1 uses CPU; >=0 uses that GPU id.
    Returns a pipeline object that outputs labels like POS/NEG/NEU depending on model.
    """
    # Use sentiment-analysis pipeline
    pipe = pipeline("sentiment-analysis", model=model_name, tokenizer=model_name, device=device)
    return pipe


def classify_news_sentiment(news_df: pd.DataFrame, pipe) -> pd.DataFrame:
    """
    Run FinBERT sentiment on each news text and return the news_df with 'label' and 'score' columns.
    label usually one of: 'positive', 'neutral', 'negative' (model dependent); we will normalize later.
    """
    texts = news_df["title"].tolist()
    # pipe can process batches; to be safe, process in small batches
    batch_size = 16
    labels = []
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        outs = pipe(batch)
        # outs: list of {'label': 'positive', 'score': 0.98} etc.
        for o in outs:
            labels.append(o["label"])
            scores.append(float(o.get("score", 0.0)))
    df = news_df.copy().reset_index(drop=True)
    df["label"] = labels
    df["score"] = scores
    return df


def aggregate_sentiment_per_company_day(news_df_with_labels: pd.DataFrame, tickers: List[str], date_index: pd.DatetimeIndex):
    """
    For each (date, ticker) compute an aggregated sentiment score.
    Strategy:
      - Map labels to numeric: positive -> +1, neutral -> 0, negative -> -1
      - Use label * score as weighted value, average across news mentioning the ticker (normalized format) on that date.
    Returns:
      sentiment_df: DataFrame indexed by date_index (dates from price_df) with columns tickers (shape T x N)
    """

    def label_to_num(lab):
        if pd.isna(lab):
            return 0.0
        s = str(lab).lower()
        if "pos" in s:
            return 1.0
        elif "neg" in s:
            return -1.0
        else:
            return 0.0

    tmp = news_df_with_labels.copy()
    #tmp["date"] = pd.to_datetime(tmp["date"]).dt.normalize()

    # Ensure mentions are parsed lists
    if isinstance(tmp["mentions"].iloc[0], str):
        tmp["mentions"] = tmp["mentions"].apply(lambda x: eval(x) if isinstance(x, str) else x)

    # Normalize tickers in 'mentions' — e.g. "AAPL.US" -> "AAPL"
    def normalize_ticker(name):
        if not isinstance(name, str):
            return name
        return name.split(".")[0].upper().strip()

    tmp["mentions"] = tmp["mentions"].apply(lambda lst: [normalize_ticker(x) for x in lst])

    # Aggregate sentiment values
    accum = {}
    for _, row in tmp.iterrows():
        d = row["date"]
        numeric = label_to_num(row["label"]) * float(row["score"])
        for t in row["mentions"]:
            if t in tickers:  # only keep known companies
                accum.setdefault((d, t), []).append(numeric)

    # Build sentiment matrix
    rows = []
    for d in date_index:
        row_vals = []
        for t in tickers:
            vals = accum.get((d, t), [])
            row_vals.append(float(np.mean(vals)) if vals else 0.0)
        rows.append(row_vals)

    sentiment_df = pd.DataFrame(rows, index=date_index, columns=tickers).astype(np.float32)
    return sentiment_df


# -------------------------
# Graph construction from news co-occurrence
# -------------------------

def normalize_ticker(name):
    if not isinstance(name, str):
        return name
    return name.split(".")[0].upper().strip()


def news_cooccurrence_graph_for_day(
    news_df: pd.DataFrame,
    tickers: List[str],
    target_date: pd.Timestamp,
    window: int = 3,
    add_self_loops: bool = True,
    min_edge_weight: float = 1e-6,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a normalized co-occurrence graph over a rolling window ending at target_date.

    Parameters:
        news_df: DataFrame with 'date' (datetime) and 'mentions' (list of tickers).
        tickers: List of valid ticker symbols (e.g., ["AAPL", "TSLA"]).
        target_date: The date for which we build the graph (use window days up to and including this date).
        window: Number of days (including target_date) to include in the co-occurrence window.
        add_self_loops: Whether to add self-loops after co-occurrence counting.
        min_edge_weight: Threshold to drop near-zero edges after normalization.

    Returns:
        edge_index: (2, E) numpy array of dtype int64.
        edge_weight: (E,) numpy array of dtype float32.
    """
    # Convert target_date to date object
    end = pd.to_datetime(target_date).date()
    start = end - pd.Timedelta(days=window - 1)

    # Filter news in the time window
    news_df['date_parsed'] = pd.to_datetime(news_df['date']).dt.date
    sub = news_df[(news_df['date_parsed'] >= start) & (news_df['date_parsed'] <= end)].copy()

    if sub.empty:
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32)

    n = len(tickers)
    idx_map = {t: i for i, t in enumerate(tickers)}

    # Initialize co-occurrence matrix
    mat = np.zeros((n, n), dtype=np.float32)

    for _, row in sub.iterrows():
        mentions = row["mentions"]
        if not isinstance(mentions, (list, tuple)):
            continue
        # Normalize and filter tickers
        valid_mentions = [normalize_ticker(m) for m in mentions if normalize_ticker(m) in idx_map]
        unique_mentions = sorted(set(valid_mentions))
        # Count pairwise co-occurrences (including self? not yet)
        for i in range(len(unique_mentions)):
            for j in range(i, len(unique_mentions)):  # include i==j for self if desired later
                a = idx_map[unique_mentions[i]]
                b = idx_map[unique_mentions[j]]
                if i == j:
                    # We'll add self-loops uniformly later; skip intra-article self-counts
                    continue
                else:
                    mat[a, b] += 1.0
                    mat[b, a] += 1.0

    # Add self-loops explicitly (standard practice in GCNs)
    if add_self_loops:
        np.fill_diagonal(mat, 1.0)  # You can also use degree or constant; 1.0 is common

    # Symmetric normalization: A_hat = D^{-1/2} A D^{-1/2}
    deg = np.sum(mat, axis=1)  # degree vector
    deg_inv_sqrt = np.where(deg > 0, deg ** (-0.5), 0.0)
    deg_inv_sqrt_matrix = np.diag(deg_inv_sqrt)
    normalized_adj = deg_inv_sqrt_matrix @ mat @ deg_inv_sqrt_matrix

    # Apply threshold to avoid storing negligible edges
    mask = normalized_adj >= min_edge_weight
    if not np.any(mask):
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32)

    # Extract upper triangle (including diagonal) to avoid duplicates
    # But since we want undirected graph, we'll get all non-zero entries and dedup via set or just use full
    rows, cols = np.nonzero(mask)
    edges = np.stack([rows, cols], axis=0).astype(np.int64)
    weights = normalized_adj[rows, cols].astype(np.float32)

    return edges, weights


# -------------------------
# Revised DynamicGraphDataset that uses news sentiment and co-occurrence edges
# -------------------------




class DynamicGraphDatasetNews(torch.utils.data.Dataset):
    """
    Dataset for dynamic graph learning using historical close prices, sentiment data,
    and technical indicators (RSI, MACD, Bollinger %B).
    Predicts next-day close prices for multiple tickers.
    """

    def __init__(
        self,
        price_df: pd.DataFrame,
        news_df: pd.DataFrame,
        sentiment_df: pd.DataFrame,
        seq_len: int = 10,
        co_window: int = 3,
        mode: str = "regression",
    ):
        assert mode in ("regression", "classification")
        self.seq_len = seq_len
        self.co_window = co_window
        self.mode = mode

        # price_df: index is DateTimeIndex
        prices = price_df.values.astype(np.float32)
        dates = pd.to_datetime(price_df.index)
        self.date_index = dates
        self.tickers = list(price_df.columns)
        T, N = prices.shape

        # Reindex sentiment to match price dates
        sentiment_df = sentiment_df.reindex(dates).fillna(0.0)
        svals = sentiment_df.values.astype(np.float32)

                # ---- PRICE NORMALIZATION  ----
        W = 252 # 1-year window for normalization; can be tuned
        T, N = prices.shape

        # Initialize arrays to the full length T
        norm_prices = np.zeros((T, N), dtype=np.float32)
        vol_norm = np.zeros((T, N), dtype=np.float32)
        targets = np.zeros((T, N), dtype=np.float32)

        # ---- PROCESSING ALL TIMESTEPS ----
        for t in range(T):
            # 1. Determine the window bounds
            # If t < W, use everything from 0 to t (Expanding)
            # If t >= W, use t-W+1 to t (Rolling)
            start_idx = max(0, t - W + 1)
            window = prices[start_idx : t + 1]

            # 2. Calculate statistics
            mean_t = window.mean(axis=0)
            std_t = window.std(axis=0) + 1e-6

            # 3. Normalize Current Price
            norm_prices[t] = (prices[t] - mean_t) / std_t

            # 4. Normalize Target Price (Price at t+1)
            # We can only do this if t < T-1
            if t < T - 1:
                targets[t] = (prices[t+1] - mean_t) / std_t

            # 5. Volatility Feature
            current_vol = window.std(axis=0)
            vol_mean = window.std(axis=0).mean()      
            vol_std = window.std(axis=0).std() + 1e-6  
            vol_norm[t] = (current_vol - vol_mean) / vol_std

        # ---- FEATURES ----
        feature_list = []
        
        for t in range(T - 1):
            feats = [
                norm_prices[t],   # Price at t
                svals[t],         # Sentiment at t
                vol_norm[t]       # Volatility at t
            ]
            feat_t = np.stack(feats, axis=1)
            feature_list.append(feat_t.astype(np.float32))

        self.features = np.stack(feature_list, axis=0)  # Shape: (T-1, N, 3)
        self.targets = targets[:T-1]                    # Shape: (T-1, N)

        # Valid indices for sequences (must have seq_len history)
        self.valid_end_idx = list(range(self.seq_len - 1, T - 1))

        # Precompute edge index & weight per timestep using news co-occurrence
        self.edge_index_list = []
        self.edge_weight_list = []
        for t in range(T - 1):
            price_date = pd.to_datetime(self.date_index[t]).date()
            ei, ew = news_cooccurrence_graph_for_day(
                news_df, self.tickers, target_date=price_date, window=self.co_window
            )
            self.edge_index_list.append(ei)
            self.edge_weight_list.append(ew)

    def __len__(self):
        return len(self.valid_end_idx)

    def __getitem__(self, idx):
        end_t = self.valid_end_idx[idx]
        start_t = end_t - (self.seq_len - 1)

        seq_feats = self.features[start_t : end_t + 1]  # (seq_len, N, F)
        seq_edge_idx = self.edge_index_list[start_t : end_t + 1]
        seq_edge_w = self.edge_weight_list[start_t : end_t + 1]
        target = self.targets[end_t]

        if self.mode == "classification":
            y = (target > 0).astype(np.longlong)  # Use longlong or int64 for classification labels
        else:
            y = target.astype(np.float32)

        sample = {
            "seq_feats": torch.from_numpy(seq_feats),
            "seq_edge_index": seq_edge_idx,
            "seq_edge_weight": seq_edge_w,
            "target": torch.from_numpy(y),
        }
        return sample

# Collate function (reuse)
def collate_dynamic(batch):
    seq_feats = torch.stack([item["seq_feats"] for item in batch], dim=0)
    targets = torch.stack([item["target"] for item in batch], dim=0)
    seq_edge_index = [item["seq_edge_index"] for item in batch]
    seq_edge_weight = [item["seq_edge_weight"] for item in batch]
    return {"seq_feats": seq_feats, "seq_edge_index": seq_edge_index, "seq_edge_weight": seq_edge_weight, "target": targets}


class TGAT(nn.Module):
    """
    GAT-GRU model (HATR-style relation-aware temporal GNN)

    - Spatial modeling: Graph Attention Network (GAT)
    - Temporal modeling: GRU
    - Graph: news co-occurrence graph
    """

    def __init__(
        self,
        in_feats: int,
        gat_hidden: int = 64,
        gru_hidden: int = 64,
        out_dim: int = 1,
        dropout: float = 0.2,
        mode: str = "regression",
        heads: int = 2,
    ):
        super().__init__()
        self.mode = mode
        self.dropout = nn.Dropout(dropout)

        # -------- Spatial (relation-aware) graph modeling --------
        # Multi-head GAT
        self.gat1 = GATConv(
            in_channels=in_feats,
            out_channels=gat_hidden // heads,
            heads=heads,
            dropout=dropout,
            concat=True,
            add_self_loops=True,
        )

        # Single-head refinement layer
        self.gat2 = GATConv(
            in_channels=gat_hidden,
            out_channels=gat_hidden,
            heads=1,
            dropout=dropout,
            concat=False,
            add_self_loops=True,
        )

        # -------- Temporal modeling --------
        self.gru = nn.GRU(
            input_size=gat_hidden,
            hidden_size=gru_hidden,
            batch_first=False,
        )

        # -------- Prediction head --------
        self.mlp = nn.Sequential(
            nn.Linear(gru_hidden, gru_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(gru_hidden // 2, out_dim),
        )

    def forward(
        self,
        seq_feats: torch.Tensor,
        seq_edge_index: List[List[np.ndarray]],
        seq_edge_weight: List[List[np.ndarray]],
    ):
        """
        seq_feats: (batch, seq_len, N, F)
        seq_edge_index[b][t]: edge_index at batch b, time t
        seq_edge_weight is intentionally unused:
        GAT learns attention weights internally.
        """

        batch, seq_len, N, F_dim = seq_feats.shape
        device = seq_feats.device

        gat_outputs = []

        # -------- Spatial attention at each time step --------
        for t in range(seq_len):
            batch_node_embeds = []
            for b in range(batch):
                x = seq_feats[b, t]  # (N, F)

                ei_np = seq_edge_index[b][t]
                if ei_np.size == 0:
                    edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
                else:
                    edge_index = torch.from_numpy(ei_np).to(device)

                # GAT ignores edge_weight by design (attention is learned)
                h = self.gat1(x, edge_index)
                h = self.dropout(F.elu(h))
                h = self.gat2(h, edge_index)
                h = F.relu(h)

                batch_node_embeds.append(h)

            gat_outputs.append(torch.stack(batch_node_embeds, dim=0))

        # (seq_len, batch, N, gat_hidden)
        seq_stack = torch.stack(gat_outputs, dim=0)

        # -------- Temporal modeling --------
        # reshape for GRU: (seq_len, batch*N, hidden)
        seq_flat = seq_stack.view(seq_len, batch * N, -1)
        gru_out, _ = self.gru(seq_flat)

        last = gru_out[-1]  # (batch*N, gru_hidden)
        preds = self.mlp(last).view(batch, N, -1)

        if self.mode == "classification":
            preds = torch.sigmoid(preds)

        return preds.squeeze(-1) if preds.shape[-1] == 1 else preds


# -------------------------------------------------
# Training / Evaluation Helpers (unchanged)
# -------------------------------------------------
def train_epoch(model, loader, optimizer, device, loss_fn):
    model.train()
    total_loss = 0.0

    for batch in loader:
        seq_feats = batch["seq_feats"].to(device)
        seq_edge_index = batch["seq_edge_index"]
        seq_edge_weight = batch["seq_edge_weight"]
        target = batch["target"].to(device)

        optimizer.zero_grad()
        preds = model(seq_feats, seq_edge_index, seq_edge_weight)
        loss = loss_fn(preds, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * seq_feats.size(0)

    return total_loss / len(loader.dataset)


def eval_epoch(model, loader, device, loss_fn, mode="regression"):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch in loader:
            seq_feats = batch["seq_feats"].to(device)
            seq_edge_index = batch["seq_edge_index"]
            seq_edge_weight = batch["seq_edge_weight"]
            target = batch["target"].to(device)

            preds = model(seq_feats, seq_edge_index, seq_edge_weight)
            loss = loss_fn(preds, target)

            total_loss += loss.item() * seq_feats.size(0)
            all_preds.append(preds.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    preds = np.concatenate(all_preds, axis=0)
    targ = np.concatenate(all_targets, axis=0)

    if mode == "classification":
        bin_preds = (preds > 0.5).astype(int)
        acc = (bin_preds == targ).mean()
        return avg_loss, {"accuracy": acc}, preds, targ
    else:
        mse = np.mean((preds - targ) ** 2)
        mae = np.mean(np.abs(preds - targ))
        return avg_loss, {"mse": mse, "mae": mae}, preds, targ



# -------------------------
# Full demo runner that ties everything together
# -------------------------
def full_news_demo(
    tickers: List[str] = None,
    start_date: str = apple_data["date"].min(),
    end_date: str = apple_data["date"].max(),
    n_news: int = len(apple_data),
    seq_len: int = 10,
    co_window: int = 3,
    batch_size: int = 16,
    epochs: int = 6,
    device_str: str = "cpu",
):
    device = torch.device(device_str)
    if tickers is None:
        # default small set (pick companies with yfinance tickers)
        tickers = ["AAPL", "AMZN", "GOOGL", "TSLA", "NFLX", "MSFT", "META", "005930.KS", "CMCSA", "IT"]

    print("Downloading price data with yfinance...")
    price_df = yf.download(tickers, start=start_date, end=end_date, progress=False)["Close"]
    # yfinance returns multi-column if multiple tickers; ensure DataFrame shape (T, N)
    if isinstance(price_df.columns, pd.MultiIndex):
        # some tickers may be missing; flatten or select 'Adj Close' level
        price_df = price_df.copy()
    price_df = price_df.dropna(how="all").ffill().dropna(axis=1)  # drop tickers with no data
    # ensure tickers order aligns
    price_df = price_df[tickers]
    print(f"Price data shape: {price_df.shape}")

    # Generate synthetic news
    print("Generating synthetic news...")
    news_df = apple_data[['date','title', 'symbols']] # Include symbols for mentions
    news_df.rename(columns={'symbols': 'mentions'}, inplace=True)
    # run FinBERT sentiment classification (this will download model the first time)
    print("Loading FinBERT pipeline (this may take a while for first run)...")
    pipe = finbert_sentiment_pipeline(model_name="yiyanghkust/finbert-tone", device=-1)  # cpu; set device=0 for GPU
    print("Classifying synthetic news with FinBERT...")
    news_labeled = classify_news_sentiment(news_df, pipe)
    news_labeled.reset_index(inplace=True, drop=True) # Reset index to make date a column again
    print("Sample labeled news:")
    print(news_labeled.head())

    # Aggregate sentiment per date-company
    print("Aggregating sentiment per company-day...")
    # create date_index for price_df
    date_index = price_df.index.date
    sentiment_df = aggregate_sentiment_per_company_day(news_labeled, list(price_df.columns), date_index)
    print("Sentiment df head:")
    print(sentiment_df.head())

    # build dataset
    print("Building DynamicGraphDatasetNews...")
    dataset = DynamicGraphDatasetNews(price_df=price_df, news_df=news_labeled, sentiment_df=sentiment_df, seq_len=seq_len, co_window=co_window, mode="regression")
    n = len(dataset)
    train_n = int(0.7 * n)
    val_n = int(0.15 * n)
    idxs = list(range(n))
    train_idx = idxs[:train_n]
    val_idx = idxs[train_n : train_n + val_n]
    test_idx = idxs[train_n + val_n :]

    from torch.utils.data import Subset, DataLoader

    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True, collate_fn=collate_dynamic)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_dynamic)
    test_loader = DataLoader(Subset(dataset, test_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_dynamic)

    in_feats = dataset.features.shape[-1]
    print(f"in_feats={in_feats}, dataset length={len(dataset)}")
    model = TGAT(in_feats=in_feats, gat_hidden=64, gru_hidden=64, out_dim=1, dropout=0.2, mode="regression").to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, device, loss_fn)
        val_loss, val_metrics, val_pred, val_targ = eval_epoch(model, val_loader, device, loss_fn, mode="regression")
        print(f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Val Metrics: {val_metrics}")

    test_loss, test_metrics, preds, targ = eval_epoch(model, test_loader, device, loss_fn, mode="regression")
    print(f"Test Loss: {test_loss:.6f} | Test Metrics: {test_metrics}")

    return model, dataset, price_df, news_labeled, sentiment_df, preds, targ